In [13]:
# Part 1 : Add date time to your campaign 
import numpy as np
import pandas as pd
from datetime import timedelta

# Load your campaign data from CSV
df = pd.read_csv("ooh_campaigns.csv")  

# Simulate campaign dates (each campaign ran for 30 days)
np.random.seed(42)

# Create a range of start dates
start_dates = pd.date_range(
    start="2024-01-01",
    end="2024-03-01",
    periods=len(df)   # Number of dates = number of campaigns
)

df["start_date"] = start_dates
df["end_date"] = df["start_date"] + timedelta(days=30)

print("✅ Dates added to campaigns!")
print(df[["campaign_name", "start_date", "end_date"]].head())


✅ Dates added to campaigns!
                   campaign_name          start_date            end_date
0  Dublin City Centre - 48 Sheet 2024-01-01 00:00:00 2024-01-31 00:00:00
1        Cork Motorway - Digital 2024-01-07 16:00:00 2024-02-06 16:00:00
2     Belfast Roadside - 6 Sheet 2024-01-14 08:00:00 2024-02-13 08:00:00
3     Dublin Airport - Supersite 2024-01-21 00:00:00 2024-02-20 00:00:00
4      Galway City - Bus Shelter 2024-01-27 16:00:00 2024-02-26 16:00:00


In [14]:
# Part 2  : Create daily perfromarmance data 
daily_data = []
for idx, campaign in df.iterrows():
    for day in range(30):

        current_date = campaign['start_date'] + timedelta(days=day)
        daily_impressions = campaign ['impressions'] / 30 + np.random.randint(-1000,1000 )
        daily_clicks = campaign['clicks'] / 30 + np.random.randint(-5,5)

        daily_data.append({
            'date' : current_date,
            'campaign_name' : campaign['campaign_name'],
            'city' : campaign['city'],
            'format' : campaign['format'],
            'daily_impressions' : max(0,daily_impressions),
            'daily_clicks' : max(0, daily_clicks)
        })

df_daily =pd.DataFrame(daily_data)

df_daily['daily_ctr'] = (df_daily['daily_clicks']/
                         df_daily['daily_impressions'] * 100
                         ).round(2)

print("Daily data created")
print(f"Total daily records: {len(df_daily)}")
print(df_daily.head(10))

Daily data created
Total daily records: 300
        date                  campaign_name    city    format  \
0 2024-01-01  Dublin City Centre - 48 Sheet  Dublin  48 Sheet   
1 2024-01-02  Dublin City Centre - 48 Sheet  Dublin  48 Sheet   
2 2024-01-03  Dublin City Centre - 48 Sheet  Dublin  48 Sheet   
3 2024-01-04  Dublin City Centre - 48 Sheet  Dublin  48 Sheet   
4 2024-01-05  Dublin City Centre - 48 Sheet  Dublin  48 Sheet   
5 2024-01-06  Dublin City Centre - 48 Sheet  Dublin  48 Sheet   
6 2024-01-07  Dublin City Centre - 48 Sheet  Dublin  48 Sheet   
7 2024-01-08  Dublin City Centre - 48 Sheet  Dublin  48 Sheet   
8 2024-01-09  Dublin City Centre - 48 Sheet  Dublin  48 Sheet   
9 2024-01-10  Dublin City Centre - 48 Sheet  Dublin  48 Sheet   

   daily_impressions  daily_clicks  daily_ctr  
0       32792.666667     39.333333       0.12  
1       32526.666667     43.333333       0.13  
2       33390.666667     40.333333       0.12  
3       33304.666667     45.333333       0.14  


In [15]:
# Aggregate daily performance across ALL campaigns
daily_totals = df_daily.groupby('date').agg({
    # For each date, sum up all daily impressions across all campaigns
    'daily_impressions': 'sum',
    # For each date, sum up all daily clicks across all campaigns
    'daily_clicks': 'sum'
}).reset_index()  # Make 'date' a regular column instead of the index

# Calculate overall CTR for each day
daily_totals['ctr'] = (
    daily_totals['daily_clicks'] /      # Total clicks on that day
    daily_totals['daily_impressions'] * # Total impressions on that day
    100                                  # Convert to percentage
).round(2)                               # Round to 2 decimal places

print("\n📊 Daily Totals (all campaigns combined):")
print(daily_totals.head(10))  # Show first 10 days



📊 Daily Totals (all campaigns combined):
                 date  daily_impressions  daily_clicks   ctr
0 2024-01-01 00:00:00       32792.666667     39.333333  0.12
1 2024-01-02 00:00:00       32526.666667     43.333333  0.13
2 2024-01-03 00:00:00       33390.666667     40.333333  0.12
3 2024-01-04 00:00:00       33304.666667     45.333333  0.14
4 2024-01-05 00:00:00       32132.666667     42.333333  0.13
5 2024-01-06 00:00:00       31996.666667     43.333333  0.14
6 2024-01-07 00:00:00       33062.666667     39.333333  0.12
7 2024-01-07 16:00:00       21621.666667     25.666667  0.12
8 2024-01-08 00:00:00       32537.666667     43.333333  0.13
9 2024-01-08 16:00:00       21561.666667     27.666667  0.13


In [16]:
# time _ Series visualisation 
import plotly.express as px 

#Create Time- Series line chart 
fig = px.line(
    daily_totals,
    x='date',
    y=['daily_impressions' , 'daily_clicks'],
    title='Campaign Perfromance Over Time',
    labels={
        'value' : 'Count',
        'date' : 'Date'
    },
    height=500
)

#Customise the layout 
fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Impressions / Clicks",
    hovermode='x unified'
)

#Display the chart 
fig.show()
# Save as HTML file
fig.write_html('time_series_performance.html')
print("✅ Time-series chart saved!")

✅ Time-series chart saved!


In [17]:
# Create time-series data grouped by city
city_daily = df_daily.groupby(['date', 'city']).agg({
    'daily_impressions': 'sum',  # Sum impressions per city per day
    'daily_clicks': 'sum'         # Sum clicks per city per day
}).reset_index()

# Calculate CTR for each city each day
city_daily['daily_ctr'] = (
    city_daily['daily_clicks'] / 
    city_daily['daily_impressions'] * 
    100
).round(2)

# Create multi-line chart (one line per city)
fig = px.line(
    city_daily,                # Use city-level daily data
    x='date',                  # X-axis = dates
    y='daily_impressions',     # Y-axis = impressions
    color='city',              # Different color for each city
    title='Daily Impressions by City Over Time',  # Chart title
    labels={
        'daily_impressions': 'Impressions',
        'date': 'Date',
        'city': 'City'
    },
    height=600
)

# Customize
fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Daily Impressions",
    hovermode='x unified'
)

# Show and save
fig.show()
fig.write_html('city_time_series.html')
print("✅ City time-series chart saved!")

✅ City time-series chart saved!
